# User Behavior Intelligence — Exploratory Data Analysis

**Dataset:** Online Retail II (UCI Machine Learning Repository)  
**Period:** December 2009 – December 2011  
**Source:** UK-based online gift retailer  

---

## Objective

This notebook explores the raw transactional dataset to understand:
1. What does the data look like, and what needs cleaning?
2. Who are the customers — how often do they buy?
3. What are the revenue and purchase patterns over time?
4. Which products drive the most revenue?
5. Are there natural customer segments we can discover?

The findings here directly inform the feature engineering and modelling pipeline in `src/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../src')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

---
## 1. Loading the Raw Data

The dataset ships as an Excel file with two sheets (Year 2009-2010 and Year 2010-2011). We load and concatenate both. Each row is a single line item on an invoice — one customer can appear on many rows within the same invoice.

In [ ]:
df_raw = pd.concat([
    pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2009-2010'),
    pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2010-2011')
], ignore_index=True)

print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()

**Initial observations:**
- ~1 million rows across the two years
- `Customer ID` has missing values — these are guest/anonymous purchases and cannot be used for customer-level analysis
- `Description` also has some nulls
- `InvoiceDate` is already datetime
- Invoices starting with `'C'` are **cancellations** — they must be removed before revenue calculations

In [ ]:
# Missing value summary
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Missing %': missing_pct})

About **24% of rows have no Customer ID**. These are anonymous transactions — we drop them for RFM analysis since we need to track individual customers over time. This is a real-world data quality issue, not a flaw in the dataset.

In [ ]:
cancellations = df_raw[df_raw['Invoice'].astype(str).str.startswith('C')]
print(f'Cancellation rows: {len(cancellations):,} ({len(cancellations)/len(df_raw)*100:.1f}% of all rows)')

---
## 2. Data Cleaning

We apply the cleaning pipeline from `src/data_cleaning.py`. The steps are:

1. Drop duplicate rows
2. Rename `Customer ID` → `CustomerID`
3. Drop rows with missing `CustomerID` or `Description`
4. Cast `CustomerID` to int then string (removes the `.0` float suffix)
5. Remove rows with `Quantity <= 0` or `Price <= 0` (returns, errors, free samples)
6. Cap `Quantity` at 10,000 — values above this are likely data entry errors
7. Remove cancellation invoices (Invoice starts with 'C')
8. Compute `TotalPrice = Quantity × Price`

In [ ]:
from data_cleaning import clean_data

df = clean_data(df_raw)
print(f'Raw rows:     {len(df_raw):,}')
print(f'Cleaned rows: {len(df):,}')
print(f'Rows removed: {len(df_raw) - len(df):,} ({(1 - len(df)/len(df_raw))*100:.1f}%)')
df.head()

In [ ]:
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Unique invoices:  {df['Invoice'].nunique():,}")
print(f"Unique products:  {df['Description'].nunique():,}")
print(f"Date range:       {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Total revenue:    GBP {df['TotalPrice'].sum():,.0f}")

After cleaning we have **~5,800 unique customers**, ~22,000 invoices, and just over **9 million GBP in total revenue** across the two years.

The large drop in row count is expected — anonymous and cancelled transactions are a normal feature of retail data.

---
## 3. Revenue Trends

Before diving into customers, let's understand the business at an aggregate level — how does revenue evolve over time?

In [ ]:
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('Month')['TotalPrice'].sum()

plt.figure(figsize=(12, 5))
plt.plot(monthly.index.astype(str), monthly.values, marker='o', color='steelblue')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (GBP)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Key observations:**
- Revenue shows a clear **seasonal spike in Q4 (October-November)** each year, consistent with Christmas gift buying
- There is a **sharp drop in December** — not a collapse in sales, but because the dataset ends mid-December 2011, cutting off the peak Christmas week
- Overall revenue trend is **upward** from 2009 to 2011, suggesting a growing customer base

This seasonality matters for the churn model — a customer who last bought in January may not be churned; they may simply be a seasonal buyer.

---
## 4. Purchase Patterns — When Do Customers Buy?

Understanding *when* orders happen helps with marketing timing.

In [ ]:
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

hourly = df.groupby('Hour')['Invoice'].nunique()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df.groupby('DayOfWeek')['Invoice'].nunique().reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(hourly.index, hourly.values, color='steelblue')
axes[0].set_title('Orders by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Unique Orders')

axes[1].bar(daily.index, daily.values, color='tomato')
axes[1].set_title('Orders by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Unique Orders')

plt.tight_layout()
plt.show()

**Key observations:**
- Orders are placed almost exclusively **during business hours (9 AM to 5 PM)**, peaking around noon
- This strongly suggests the customer base is **B2B (businesses buying wholesale)** rather than individual consumers
- **Thursday and Tuesday** are the busiest days; **Sunday has almost zero orders** — further B2B evidence
- This matters for churn interpretation — a business customer not ordering for 60 days is very different from an individual consumer

---
## 5. Top Products by Revenue

In [ ]:
exclude = ['POSTAGE', 'MANUAL', 'DOTCOM POSTAGE', 'CRUK COMMISSION']
df_products = df[~df['Description'].str.upper().isin(exclude)]

top10 = df_products.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
top10.plot(kind='barh', color='steelblue')
plt.title('Top 10 Products by Total Revenue')
plt.xlabel('Total Revenue (GBP)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
print(top10)

The top revenue products are mostly **decorative home and gift items** — consistent with a UK gift wholesaler. A small number of products contribute disproportionately to revenue (Pareto principle), which means recommending these to new customers is a safe default strategy.

---
## 6. RFM Feature Engineering

RFM stands for **Recency, Frequency, Monetary** — three dimensions that together describe a customer's value and engagement.

| Feature | Definition | Why it matters |
|---|---|---|
| **Recency** | Days since last purchase | Recent buyers are more likely to respond to offers |
| **Frequency** | Number of unique invoices | Frequent buyers are more loyal |
| **Monetary** | Total spend | High spenders drive most revenue |

We compute these relative to a **reference date** = one day after the last invoice in the dataset. Recency = 1 means the customer bought on the last possible day.

In [ ]:
from analytics import compute_rfm

rfm = compute_rfm(df)
rfm.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes, ['recency', 'frequency', 'monetary'], ['steelblue', 'tomato', 'green']):
    data = rfm[col].clip(upper=rfm[col].quantile(0.99))
    ax.hist(data, bins=40, color=color, edgecolor='white')
    ax.set_title(f'{col.capitalize()} Distribution')
    ax.set_xlabel(col.capitalize())
    ax.set_ylabel('Customers')

plt.suptitle('RFM Feature Distributions (clipped at 99th percentile)', y=1.02)
plt.tight_layout()
plt.show()

**Key observations:**
- All three distributions are heavily **right-skewed** — most customers are low-frequency, low-spend, but a long tail of high-value customers exists
- This skewness is why we **cap at the 99th percentile and apply StandardScaler** before K-Means — the algorithm is sensitive to scale and outliers
- The extreme monetary outliers are likely wholesale/B2B accounts placing very large orders

---
## 7. RFM Segments

Each customer gets an R, F, and M score from 1 to 5 (quintile-based). The sum (3 to 15) maps to a segment label.

In [ ]:
seg_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(8, 5))
seg_counts.plot(kind='bar', color=['gold', 'steelblue', 'tomato', 'grey'])
plt.title('RFM Segment Distribution')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(seg_counts)
print('\nThresholds: RFM >= 12 = Champions | >= 9 = Loyal | >= 6 = At Risk | < 6 = Lost')

A high proportion of 'At Risk' and 'Lost' customers is **normal for retail datasets** — most customers buy once or twice and never return. The goal is to identify Champions and Loyal customers who drive bulk revenue, and design retention strategies for At Risk customers before they become Lost.

---
## 8. K-Means Clustering

RFM scoring gives rule-based segments. K-Means lets the **data itself** decide where the natural groupings are — without pre-defining score thresholds.

We use the **Elbow Method** to choose k: plot inertia (sum of squared distances to cluster centres) for k=1 to 10 and look for where the curve 'elbows'. We also compute **Silhouette Score** — closer to 1 means clusters are well-separated.

In [ ]:
from segmentation import plot_elbow
plot_elbow(rfm)

In [ ]:
from segmentation import fit_clusters, visualize_clusters, compute_churn_risk
rfm_clustered, X_scaled = fit_clusters(rfm, k=3)

The three clusters are labelled **automatically** based on the cluster means — no hardcoded mapping:
- **High Value** — highest average monetary spend (wholesale/bulk buyers)
- **Active Regular** — moderate recency and frequency (core retail customers)
- **Churned** — highest recency (bought longest ago), lowest engagement

In [ ]:
visualize_clusters(rfm_clustered, X_scaled)

The PCA plot reduces the 3D RFM space to 2D for visualisation. Visually separated clusters confirm that K-Means found meaningful groups rather than arbitrary partitions.

---
## 9. Churn Risk Scoring

We estimate churn risk using min-max normalised recency. Score of **1.0** = bought longest ago = highest churn risk. Score of **0.0** = bought most recently.

In [ ]:
rfm_clustered = compute_churn_risk(rfm_clustered)

plt.figure(figsize=(10, 5))
for name, group in rfm_clustered.groupby('Cluster_Name'):
    group['Churn_Risk'].hist(bins=30, alpha=0.6, label=name)
plt.title('Churn Risk Distribution by Cluster')
plt.xlabel('Churn Risk Score')
plt.ylabel('Number of Customers')
plt.legend()
plt.tight_layout()
plt.show()

As expected, the **Churned** cluster has the highest average churn risk scores. This validates that K-Means labels are meaningful — the clusters align with the recency-based churn signal even though recency was just one of three inputs to the clustering.

---
## 10. Cluster-Aware Recommendations

Instead of recommending the same popular products to everyone, we recommend products popular **within each customer's cluster**. A High Value wholesale buyer should see different recommendations than an Active Regular retail buyer.

In [ ]:
from recommendation import recommend_for_customer, recommend_popular

print('=== Global Popular Products ===')
print(recommend_popular(df))

for cluster_name in rfm_clustered['Cluster_Name'].unique():
    sample_customer = rfm_clustered[rfm_clustered['Cluster_Name'] == cluster_name].index[0]
    print(f'\n--- {cluster_name} (sample: customer {sample_customer}) ---')
    print(recommend_for_customer(sample_customer, df, rfm_clustered))

The per-cluster recommendations differ meaningfully — High Value customers see bulk/wholesale-type items while Active Regular customers see popular individual gift items. This is the core business value of the segmentation.

---
## 11. Summary

| Finding | Implication |
|---|---|
| ~24% of transactions are anonymous | Guest checkout reduces CRM coverage |
| Ordering peaks 10 AM–3 PM on weekdays | Customer base is predominantly B2B |
| Clear Q4 revenue spike | Seasonal inventory and marketing planning needed |
| 3 natural customer clusters (k=3) | High Value, Active Regular, Churned |
| Churned cluster has highest recency | Target with re-engagement campaigns |
| Cluster-specific recommendations differ | Generic recommendations leave revenue on the table |